# Notebook 1 — Data Ingestion and Storage Design
**Problem:** Kickstarter campaign success prediction (binary classification: state=1 success, state=0 failed)

This notebook covers:
- SparkSession configuration with justification
- CSV ingestion with data validation
- Partitioning strategy and Parquet storage
- Broadcast join implementation
- Memory management (persist / unpersist)

## 1. SparkSession Configuration
Configuration choices are justified below and documented fully in `config/spark_config.yaml`.

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, count, when, isnan, isnull

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Kickstarter - Data Ingestion") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024)) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark version: 3.5.5
Driver memory: 4g
Shuffle partitions: 50


**SparkSession Config Justification:**
| Config | Value | Reason |
|--------|-------|--------|
| `driver.memory` | 4g | Dataset ~150k rows; driver collects evaluation results |
| `shuffle.partitions` | 50 | Avoids over-partitioning in local mode (~3k rows/partition at 50 partitions) |
| `adaptive.enabled` | true | AQE auto-coalesces small partitions post-shuffle |
| `autoBroadcastJoin` | 10 MB | Country/category lookup tables < 10 MB → broadcast join |

## 2. Data Ingestion from CSV

In [2]:
CSV_PATH    = "../data/kickstarter_2022-2021_unique_blurbs.csv"
PARQUET_OUT = "../data/parquet/kickstarter"

df_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .option("quote",  '"') \
    .csv(CSV_PATH)

print("Raw row count :", df_raw.count())
print("Columns       :", len(df_raw.columns))
df_raw.printSchema()

Raw row count : 203341
Columns       : 41
root
 |-- backers_count: integer (nullable = true)
 |-- blurb: string (nullable = true)
 |-- category: string (nullable = true)
 |-- converted_pledged_amount: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- country_displayable_name: string (nullable = true)
 |-- created_at: integer (nullable = true)
 |-- creator: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- currency_symbol: string (nullable = true)
 |-- currency_trailing_code: boolean (nullable = true)
 |-- current_currency: string (nullable = true)
 |-- deadline: integer (nullable = true)
 |-- disable_communication: boolean (nullable = true)
 |-- friends: string (nullable = true)
 |-- fx_rate: double (nullable = true)
 |-- goal: double (nullable = true)
 |-- id: integer (nullable = true)
 |-- is_backing: boolean (nullable = true)
 |-- is_starrable: boolean (nullable = true)
 |-- is_starred: boolean (nullable = true)
 |-- launched_at: integer (null

## 3. Data Validation at Ingestion

In [3]:
# Check null counts for key columns
key_cols = ["category", "country", "goal", "state", "launched_at", "deadline", "staff_pick"]

dtypes = dict(df_raw.dtypes)

numeric_cols = [c for c in key_cols if dtypes.get(c) not in ("string", "boolean")]
string_cols  = [c for c in key_cols if dtypes.get(c) == "string"]
bool_cols    = [c for c in key_cols if dtypes.get(c) == "boolean"]

null_counts = df_raw.select(
    [count(when(isnull(c) | isnan(c), c)).alias(c) for c in numeric_cols] +
    [count(when(isnull(c), c)).alias(c) for c in string_cols] +
    [count(when(isnull(c), c)).alias(c) for c in bool_cols]
)
null_counts.show()

+----+-----+-----------+--------+--------+-------+----------+
|goal|state|launched_at|deadline|category|country|staff_pick|
+----+-----+-----------+--------+--------+-------+----------+
|   0|    0|          0|       0|       0|      0|         0|
+----+-----+-----------+--------+--------+-------+----------+



In [4]:
# Validation rules:
#   1. goal must be positive (no free campaigns)
#   2. state must be binary (0=failed, 1=success)
#   3. Drop rows with null key columns

df_validated = df_raw \
    .filter(col("goal") > 0) \
    .filter(col("state").isin(0, 1)) \
    .dropna(subset=["category", "country", "launched_at", "deadline"])

dropped = df_raw.count() - df_validated.count()
print(f"Rows after validation: {df_validated.count()} (dropped {dropped})")

Rows after validation: 203341 (dropped 0)


In [5]:
# Class distribution check — important for classification imbalance
df_validated.groupBy("state").count().show()
print("Countries:", df_validated.select("country").distinct().count())
print("Categories:", df_validated.select("category").distinct().count())

+-----+------+
|state| count|
+-----+------+
|    0| 86825|
|    1|116516|
+-----+------+

Countries: 25
Categories: 171


## 4. Broadcast Join — Country Metadata Enrichment

In [8]:
import os
from pyspark.sql.functions import col, broadcast

# Checkpoint dir — breaks full DAG lineage
CHECKPOINT_DIR = "../output/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)

# ── Guard: rebuild df_validated if run out of order ───────────────────────────
try:
    _ = df_validated
    print("df_validated found in session")
except NameError:
    print("df_validated missing — rebuilding ...")
    try:
        _ = df_raw
    except NameError:
        CSV_PATH = "../data/kickstarter_2022-2021_unique_blurbs.csv"
        df_raw = spark.read \
            .option("header", True).option("inferSchema", True) \
            .option("multiLine", True).option("escape", '"').option("quote", '"') \
            .csv(CSV_PATH)
    df_validated = df_raw \
        .filter(col("goal") > 0) \
        .filter(col("state").isin(0, 1)) \
        .dropna(subset=["category", "country", "launched_at", "deadline"])

# Checkpoint df_validated first (already done if cell ran before, safe to repeat)
df_validated = df_validated.checkpoint(eager=True)
print(f"df_validated: {df_validated.count()} rows")

# ── Region lookup via pure SQL VALUES — no Python worker involved ─────────────
# spark.sql() VALUES is 100% JVM; createDataFrame(python_list) routes through
# PythonRunner and causes EOFException on Windows local mode.
region_df = spark.sql("""
    SELECT * FROM VALUES
        ('US','North America'),('CA','North America'),('MX','North America'),
        ('GB','Europe'),('DE','Europe'),('FR','Europe'),('IT','Europe'),
        ('ES','Europe'),('NL','Europe'),('SE','Europe'),('NO','Europe'),
        ('DK','Europe'),('CH','Europe'),('BE','Europe'),('AT','Europe'),
        ('IE','Europe'),('PL','Europe'),('LU','Europe'),('SI','Europe'),('GR','Europe'),
        ('AU','Oceania'),('NZ','Oceania'),
        ('JP','Asia'),('HK','Asia'),('SG','Asia')
    AS t(country, region)
""")

df_enriched = df_validated.join(broadcast(region_df), on="country", how="left")

# Checkpoint enriched df — pure JVM join plan now, no Python worker
df_enriched = df_enriched.checkpoint(eager=True)
print(f"Enriched row count: {df_enriched.count()}")

# Print mapping without any Spark action
region_data = [
    ("AT","Europe"),("BE","Europe"),("CH","Europe"),("DE","Europe"),("DK","Europe"),
    ("ES","Europe"),("FR","Europe"),("GB","Europe"),("GR","Europe"),("IE","Europe"),
    ("IT","Europe"),("LU","Europe"),("NL","Europe"),("NO","Europe"),("PL","Europe"),
    ("SE","Europe"),("SI","Europe"),
    ("HK","Asia"),("JP","Asia"),("SG","Asia"),
    ("CA","North America"),("MX","North America"),("US","North America"),
    ("AU","Oceania"),("NZ","Oceania"),
]
print("\nCountry → Region mapping:")
for country, region in region_data:
    print(f"  {country:<4} → {region}")


df_validated found in session
df_validated: 203341 rows
Enriched row count: 203341

Country → Region mapping:
  AT   → Europe
  BE   → Europe
  CH   → Europe
  DE   → Europe
  DK   → Europe
  ES   → Europe
  FR   → Europe
  GB   → Europe
  GR   → Europe
  IE   → Europe
  IT   → Europe
  LU   → Europe
  NL   → Europe
  NO   → Europe
  PL   → Europe
  SE   → Europe
  SI   → Europe
  HK   → Asia
  JP   → Asia
  SG   → Asia
  CA   → North America
  MX   → North America
  US   → North America
  AU   → Oceania
  NZ   → Oceania


**Broadcast join justification:**  
The region lookup table has ~25 rows (~1 KB). Broadcasting it eliminates a shuffle join that would otherwise move the entire dataset across the network. This is the standard pattern for dimension-table enrichment in distributed pipelines.

## 5. Persist → Write Parquet → Unpersist

In [9]:
# df_enriched is already checkpointed — no additional persist needed
# Write Parquet partitioned by country
df_enriched.write \
    .mode("overwrite") \
    .partitionBy("country") \
    .parquet(PARQUET_OUT)

print("Parquet written to:", PARQUET_OUT)
print("Done.")


Parquet written to: ../data/parquet/kickstarter
Done.


In [10]:
# Verify Parquet partitions on disk
import os

parquet_root = "../data/parquet/kickstarter"
partitions   = [d for d in os.listdir(parquet_root) if d.startswith("country=")]
print(f"Partition directories ({len(partitions)}):")
for p in sorted(partitions):
    files = os.listdir(os.path.join(parquet_root, p))
    parquet_files = [f for f in files if f.endswith(".parquet")]
    print(f"  {p}: {len(parquet_files)} parquet file(s)")

Partition directories (25):
  country=AT: 1 parquet file(s)
  country=AU: 1 parquet file(s)
  country=BE: 1 parquet file(s)
  country=CA: 1 parquet file(s)
  country=CH: 1 parquet file(s)
  country=DE: 1 parquet file(s)
  country=DK: 1 parquet file(s)
  country=ES: 1 parquet file(s)
  country=FR: 1 parquet file(s)
  country=GB: 1 parquet file(s)
  country=GR: 1 parquet file(s)
  country=HK: 1 parquet file(s)
  country=IE: 1 parquet file(s)
  country=IT: 1 parquet file(s)
  country=JP: 1 parquet file(s)
  country=LU: 1 parquet file(s)
  country=MX: 1 parquet file(s)
  country=NL: 1 parquet file(s)
  country=NO: 1 parquet file(s)
  country=NZ: 1 parquet file(s)
  country=PL: 1 parquet file(s)
  country=SE: 1 parquet file(s)
  country=SG: 1 parquet file(s)
  country=SI: 1 parquet file(s)
  country=US: 1 parquet file(s)


In [11]:
# Verify read-back from Parquet — predicate pushdown in action
df_check = spark.read.parquet(PARQUET_OUT)
print("Read-back row count:", df_check.count())

# This query only reads country=US partition (partition pruning)
us_count = df_check.filter(col("country") == "US").count()
print("US campaigns:", us_count)

spark.stop()
print("SparkSession stopped.")

Read-back row count: 203341
US campaigns: 139078
SparkSession stopped.
